In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

### Load classes to understand the model .pt files

In [2]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model=128, n_heads=4, d_ff=512, dropout=0.1):
        super().__init__()

        # Normalization and creating heads
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

        # Drops 10% of the residual connection
        self.residual_dropout = nn.Dropout(dropout)

    def forward(self, h, causal_mask, padding_mask):
        # Attention sublayer
        normalized = self.norm1(h)
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=causal_mask,
            key_padding_mask=padding_mask,
            need_weights=False,
        )
        h = h + self.residual_dropout(attended)

        # Feed-forward sublayer
        h = h + self.residual_dropout(
            self.feed_forward(self.norm2(h))
        )
        return h

In [3]:
class CausalProteinTransformerClassifier(nn.Module):
    def __init__(
        self,
        vocab_size=23,
        max_length=129,
        d_model=128,
        n_heads=4,
        n_layers=4,
        d_ff=512,
        dropout=0.1,
        pad_id=0,
    ):
        super().__init__()
        self.pad_id = pad_id

        self.token_embedding = nn.Embedding(
            vocab_size, d_model, padding_idx=pad_id
        )
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.embedding_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, 3) # Down to the 3 families


    def forward(self, x):
        _, T = x.shape

        positions = torch.arange(T, device=x.device)
        h = self.token_embedding(x) + self.position_embedding(positions)
        h = self.embedding_dropout(h)

        causal_mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool, device=x.device),
            diagonal=1,
        )
        padding_mask = x.eq(self.pad_id)

        for block in self.blocks:
            h = block(h, causal_mask, padding_mask)

        #return self.output(self.final_norm(h))
        return self.output(self.final_norm(h)) # YOU CANNOT INTRODUCE SOFTMAX HERE AS IT IS INTEGRATED IN THE LOSS-FUNCTION LATER F.softmax(self.output(self.final_norm(h)), dim=2) # F.softmax(x, dim=2) # softmax to be performed on the third dimension


In [4]:
class CausalProteinTransformer(nn.Module):
    def __init__(
        self,
        vocab_size=23,
        max_length=129,
        d_model=128,
        n_heads=4,
        n_layers=4,
        d_ff=512,
        dropout=0.1,
        pad_id=0,
    ):
        super().__init__()
        self.pad_id = pad_id

        self.token_embedding = nn.Embedding(
            vocab_size, d_model, padding_idx=pad_id
        )
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.embedding_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        _, T = x.shape

        positions = torch.arange(T, device=x.device)
        h = self.token_embedding(x) + self.position_embedding(positions)
        h = self.embedding_dropout(h)

        causal_mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool, device=x.device),
            diagonal=1,
        )
        padding_mask = x.eq(self.pad_id)

        for block in self.blocks:
            h = block(h, causal_mask, padding_mask)

        return self.output(self.final_norm(h))

### Load the model that predicts family:

In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")  # Apple Silicon GPU
else:
    device = torch.device("cpu")

# OBS: !!!!!!!!!!!!!!!!!!
device = torch.device("cpu") # This is faster than my MPS. The issue is my bad loops.

In [6]:
model_predictor = CausalProteinTransformerClassifier(
    vocab_size=23,
    max_length=129,
    pad_id=0,
)  # Use the same class and arguments as during training
model_predictor.load_state_dict(torch.load("classifierModel.pt", weights_only=True))
model_predictor = model_predictor.to(device)
model_predictor.eval()

CausalProteinTransformerClassifier(
  (token_embedding): Embedding(23, 128, padding_idx=0)
  (position_embedding): Embedding(129, 128)
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (attention): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (feed_forward): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=512, out_features=128, bias=True)
      )
      (residual_dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (final_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (output): Linear(in_features=128, out_features

### Load the model that kan generate a protein sequence:

In [7]:
model_generator = CausalProteinTransformer(
    vocab_size=23,
    max_length=129,
    pad_id=0,
)  # Use the same class and arguments as during training
model_generator.load_state_dict(torch.load("transformer_best.pt", weights_only=True))
model_generator = model_generator.to(device)
model_generator.eval()

CausalProteinTransformer(
  (token_embedding): Embedding(23, 128, padding_idx=0)
  (position_embedding): Embedding(129, 128)
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (attention): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (feed_forward): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=512, out_features=128, bias=True)
      )
      (residual_dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (final_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (output): Linear(in_features=128, out_features=23, bias=

### Make code where the generator predicts the next token in a sequence,
### then the predictor calculates the family propabilites for all the different combinations, 
### then the 2 probabilites are added together (using hyper-parameter \lambda),
### given a certain desired faimly, choose what token should be the next in the sequence.
### Loop over that above untill reaching 129 tokens, or token 2 (<EOS>) is chosen.

In [8]:
# Create reverse mapping: index -> token
from protgen.data import ProteinDataset
train_data = ProteinDataset(path_to_data="../data/train.csv")
idx_to_token = {v: k for k, v in train_data.vocab.items()}

In [9]:
# Choose family:
#       chosen_family  =  0 -> AcyP
#       chosen_family  =  1 -> HPr
#       chosen_family  =  2 -> HU
chosen_family  =  0

In [10]:
import csv
# I do not make the calculation for token 0 and 1, as it is the padding and <BOS> which can not exist inside a sequence.
possible_next_tokens = np.arange(2,23) # The values from 2 to 22 (a total of 21 values as "0" and "1" are not included)
# Make the sequence generator stop when the length of the seqeucen reaches 129.
max_length = 129
number_of_seqeunces_wanted = 1000


for chosen_family in [1, 2]: # Skipping family 0 as I have already run the code with that.
    lambda_ranges = [0.005, 0.01, 0.05, 0.1, 1., 10.]

    for regularisation_lambda in lambda_ranges:
        print("Starting in loop with regularisation_lambda equal", regularisation_lambda)
        multiple_seqeunces_loop_val = 1 # I know it hurt that I start with 1 and not 0, but we start with the first (1st) loop. 
        multiple_seq = [[]]
        while(multiple_seqeunces_loop_val <= number_of_seqeunces_wanted):
            # Init sequence
            device = next(model_generator.parameters()).device
            single_seq = torch.tensor([[1]], device=device)
            for single_sequence_loop in range(max_length-1): # Minus one, as we have already started the sequence with one token (<BOS> 1)
                ### GENERATOR
                # Get generator predtions (the output is not a probability distibution, as the softmax is first introduced in the loss fundtion (F.cross_entropy)):
                generator_output = model_generator(single_seq)
                generator_prop   = F.softmax(generator_output[:,-1,:], dim=1).squeeze() # Only the last row (the next prediction for a token). Squeeze to go from 2D to 1D.

                ### PREDICTOR
                # Make loop that goes over all the possible seqeunces that can be created with one more token and calculate probability distributions for which families their belong to:
                for next_token in possible_next_tokens:
                    #pos_single_seq = torch.cat([single_seq, torch.tensor([[next_token]])], dim=1)
                    pos_single_seq = torch.cat([single_seq, single_seq.new_tensor([[next_token]])], dim=1) # Keep on correct device (MPS for me)
                    # Get predictor predtions (the output is not a probability distibution, as the softmax is first introduced in the loss fundtion (F.cross_entropy)):
                    predictor_output = model_predictor(pos_single_seq)
                    predictor_prop   = F.softmax(predictor_output[:,-1,:], dim=1) # Only the last row (family guess knowing the whole (partial) seqeunce)
                    # Save the distribution for this one possible (partial) sequence:
                    if next_token == possible_next_tokens[0]: # If first loop create variable to hold the different probability distributinos
                        collected_predictor_prop = predictor_prop
                    else:
                        collected_predictor_prop = torch.cat([collected_predictor_prop, predictor_prop])

                ### COLLAPSE PREDICTOR DISTRIBUTIONS ACCORDING TO THE WANTED FAMILY
                collected_predictor_prop_family = collected_predictor_prop[:,chosen_family]

                ### ADDING THE PROBABILITY DISTRIBUTIONS
                # I remove the probabilites for padding (0) and <BOS> (1) as they are not possible to have in a sequence
                final_prop_dist = generator_prop[2:] + regularisation_lambda * collected_predictor_prop_family

                ### CHOSE TOKEN, BUILD SEQUENCE, AND CONTINUE LOOP
                # Use the final probability distribution to draw a sample that will be the next token:
                chosen_token = torch.multinomial(final_prop_dist, num_samples=1) + 2 # Added two because we have removed 0 and 1 from the vectors, so if the first index (index zero) in final_prop_dist is actually index 2.
                if chosen_token == 2: # If the token is <EOS>, end the loop and start on a new sequence. We do not add the <EOS> to the sequence as it then just has to removed later when converting from indexes to letters.
                    break
                #single_seq = torch.cat([single_seq, torch.tensor([[chosen_token]])], dim=1)
                single_seq = torch.cat([single_seq, single_seq.new_tensor([[chosen_token]])], dim=1) # Keep on correct device (MPS for me)

            ### MAKE SEQUENCE FROM ARRAY TO STRING WITH LETTERS AND HAVE THEM
            if single_seq.shape[1] != 1: # If it is equal to 1, then the first token that where predicted was the <EOS>. So no protein has been created and this run should be done again.
                single_seq_tokens = [idx_to_token[i] for i in single_seq.squeeze().tolist()[1:]] # Make single_seq from 2D to 1D and then to a list. [1:] is to remove the <BOS> token
                multiple_seq.append("".join(single_seq_tokens))
                multiple_seqeunces_loop_val += 1 # One more sucessfull protein sequence has been made
            # Progress tracker:
            #print(multiple_seqeunces_loop_val)
            if np.mod(multiple_seqeunces_loop_val,10) == 0:
                print("Regularisation_lambda equal", regularisation_lambda)
                print("In that overall loop, the number of created sequences are", multiple_seqeunces_loop_val)
        multiple_seq = multiple_seq[1:] # Remove the first black sequence this variable was initalised with.
        print("Test to see how the sequences look like:")
        print(multiple_seq)
        
        ### Make the CSV file:
        with open(f'../results/sequences/generated_sequences_phase2guided_lambda{str(int(regularisation_lambda*1000))}_family{str(chosen_family)}.csv', 'w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(['sequence'])
            for seq in multiple_seq:
                writer.writerow([seq])

Starting in loop with regularisation_lambda equal 0.005
Regularisation_lambda equal 0.005
In that overall loop, the number of created sequences are 10
Regularisation_lambda equal 0.005
In that overall loop, the number of created sequences are 20
Regularisation_lambda equal 0.005
In that overall loop, the number of created sequences are 30
Regularisation_lambda equal 0.005
In that overall loop, the number of created sequences are 40
Regularisation_lambda equal 0.005
In that overall loop, the number of created sequences are 50
Regularisation_lambda equal 0.005
In that overall loop, the number of created sequences are 60
Regularisation_lambda equal 0.005
In that overall loop, the number of created sequences are 70
Regularisation_lambda equal 0.005
In that overall loop, the number of created sequences are 80
Regularisation_lambda equal 0.005
In that overall loop, the number of created sequences are 90
Regularisation_lambda equal 0.005
In that overall loop, the number of created sequences a

In [11]:
print("12.30 - 940")

12.30 - 940
